# MOSTA mouse organogenesis

This notebook shows the complete `mosta` analysis: data preparation,
model training, downstream analysis, and the commands used for the paper
figures. Edit the paths in **Setup** before starting a run.

## Setup

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)
DATASET_CONFIG = 'mosta'
RAW_H5AD = Path("data/mosta_raw.h5ad")
OUTPUT_DIR = Path("tutorial_outputs/mosta")
ALIGNED_H5AD = OUTPUT_DIR / "preprocess" / 'mosta_aligned.h5ad'
MODEL_DIR = OUTPUT_DIR / "training"


RUN_PREPARATION = False
RUN_PREPROCESS_AND_TRAIN = False
RUN_DOWNSTREAM = False

In [2]:
config, config_source = load_workflow_config(DATASET_CONFIG)
dataset = config["dataset"]
scientific = config["scientific"]
downstream = config["downstream"]

pd.DataFrame(
    {
        "setting": [
            "dataset",
            "configuration",
            "raw time column",
            "cell annotation",
            "observed training times",
            "classifier neighbors",
        ],
        "value": [
            dataset["display_name"],
            config_source,
            config["preprocess"]["time_key"],
            dataset["annotation_key"],
            ", ".join(map(str, downstream["observed"])),
            scientific["classifier_k"],
        ],
    }
)

,setting,value
0,dataset,MOSTA mouse organogenesis
1,configuration,example configuration: mosta
2,raw time column,timepoint
3,cell annotation,Annotation
4,observed training times,"0.0, 1.0, 2.0, 3.0"
5,classifier neighbors,10


## Data preparation

The dataset configuration records the count layer, time mapping, spatial
coordinates, and alignment settings. The command below reads the raw H5AD and
writes the aligned H5AD and edge model used for training.

### 1. preprocess

```text
cytobridge workflow --config mosta --step preprocess --input-h5ad <raw.h5ad> --output-dir <run>
```

Input: `raw H5AD and the dataset configuration`

Creates: `<run>/preprocess/mosta_aligned.h5ad; <run>/preprocess/edge_classifier/mosta_edge_model.pt; preprocessing records`

Continue with: `training`

In [3]:
preparation_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess",),
)
preparation_plan = build_workflow_plan(
    config,
    source=config_source,
    options=preparation_options,
)
print(render_workflow_plan(preparation_plan))

CytoBridge workflow plan
dataset: MOSTA mouse organogenesis (mosta)
config: example configuration: mosta
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/mosta/preprocess/mosta_aligned.h5ad
    edge predictor: not requested during preprocessing
  train: skipped; add --train to run (GPU required for training)
  downstream: skipped (GPU recommended)


In [4]:
if RUN_PREPARATION:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before preprocessing: {RAW_H5AD}")
    preparation_result = run_workflow(config, options=preparation_options)
    preparation_result
else:
    print("Data preparation is off. Set RUN_PREPARATION = True to run it.")

Data preparation is off. Set RUN_PREPARATION = True to run it.


## Training

The full run starts from the raw H5AD, writes the aligned data, fits the
interaction edge model when needed, and trains CytoBridge. Training requires a
CUDA-capable environment.

### 1. preprocess and train

```text
cytobridge workflow --config mosta --step preprocess --step train --train --input-h5ad <raw.h5ad> --output-dir <run> --device cuda
```

Input: `raw H5AD, dataset configuration, and LR database`

Creates: `<run>/training/<stage>/best_model.pth or score_model.pth; <run>/training/adata.h5ad; training_history.csv; training_run_summary.json`

Continue with: `downstream`

In [5]:
training_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess", "train"),
    train=True,
)
training_plan = build_workflow_plan(
    config,
    source=config_source,
    options=training_options,
)
print(render_workflow_plan(training_plan))

CytoBridge workflow plan
dataset: MOSTA mouse organogenesis (mosta)
config: example configuration: mosta
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/mosta/preprocess/mosta_aligned.h5ad
    edge predictor: will be trained automatically
      graph database: package: CytoBridge/workflow_databases/CellChatDB.ligrec.mouse.csv
      database source: included CellChatDB resource
      interaction cutoff: 0.02400244047956264
      decision threshold source: validation-selected during de novo training
      output: tutorial_outputs/mosta/preprocess/edge_classifier/mosta_edge_model.pt
  train: ready (GPU required for training)
    training config: mosta_spatial_full_alpha_express_0015.yaml
    interaction cutoff: 0.02400244047956264
    edge predictor threshold source: validation-selected during preprocessing
    edge predictor: tutorial_outputs/mosta/preprocess/ed

In [6]:
if RUN_PREPROCESS_AND_TRAIN:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before training: {RAW_H5AD}")
    training_result = run_workflow(config, options=training_options)
    training_result
else:
    print("Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.")

Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.


## Downstream analysis

Downstream analysis reads the aligned H5AD and fitted model from the training
directory. It writes generated states, velocity, growth, composition,
communication, ligand–receptor tables, and standard figures.

### 1. downstream

```text
cytobridge workflow --config mosta --step downstream --aligned-h5ad <run>/preprocess/mosta_aligned.h5ad --model-dir <run>/training --output-dir <run>
```

Input: `aligned H5AD; <run>/training; dataset-matched LR database`

Creates: `<run>/downstream/summary.json; slice_data/*.h5ad; velocity/velocity_components.npz; growth/growth_by_cell.csv; composition/celltype_composition.csv; communication and ligand_receptor tables; standard figures`

Continue with: `paper-specific continuation shown in the paper-figure notebook`

In [7]:
downstream_options = WorkflowOptions(
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    output_dir=OUTPUT_DIR,
    steps=("downstream",),
)
downstream_plan = build_workflow_plan(
    config,
    source=config_source,
    options=downstream_options,
)
print(render_workflow_plan(downstream_plan))

CytoBridge workflow plan
dataset: MOSTA mouse organogenesis (mosta)
config: example configuration: mosta
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: skipped (GPU for spatial alignment)
  train: skipped; add --train to run (GPU required for training)
  downstream: ready (GPU recommended for SDE simulation and classifier fitting)
    model format: current
    output: tutorial_outputs/mosta/downstream
    generated states: observed times=[0.0, 1.0, 2.0, 3.0], additional times=[0.25, 0.5, 0.75, 1.25, 1.5, 1.75, 2.25, 2.5, 2.75]
      simulation settings: dt=0.05, sigma=0.03, daughter noise=0, growth alpha=1
    interpolation and classification: enabled
    time-slice velocity: enabled
    growth: enabled when present in the model
    cell-type composition: enabled
    sparse communication: enabled
    standard figures: enabled
      note: snapshots, mosaic, growth, composition, and velocity; 3D communication only when the model has a

In [8]:
if RUN_DOWNSTREAM:
    missing = [path for path in (ALIGNED_H5AD, MODEL_DIR) if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing aligned data or model directory: {missing}")
    downstream_result = run_workflow(config, options=downstream_options)
    downstream_result
else:
    print("Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.")

Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.


## Paper figures

Continue with these commands to calculate the values used in the paper. Each
step states which downstream files it reads and which paper notebook uses its
output.

- [Main Figure 4](../paper_figures/main_figure_4.ipynb)
- [Supplementary Figures S11–S18](../paper_figures/mosta_figures.ipynb)

### 1. calculate MOSTA panel inputs

Used for: Main Figure 4; S11-S18

```text
cytobridge workflow --config mosta --step downstream --aligned-h5ad <run>/preprocess/mosta_aligned.h5ad --model-dir <run>/training --output-dir <run>
```

Input: `aligned MOSTA H5AD and six-stage model used for the paper`

Creates: `global-t0 states; growth; composition; lineage; gene-program; GO and LR tables`

Continue with: `run the calculation_scripts recorded in the MOSTA release figure_index.csv`

### 2. draw and assemble the figure pages

Used for: Main Figure 4; S11-S18

```text
release_artifacts/mosta_package_native_corrected_20260826_v1/reproduction/main_figure4_complete plus figure_index.csv renderers
```

Input: `figure-specific numerical tables from the downstream run`

Creates: `five Main Figure 4 vector panels and eight SI vector pages`

Continue with: `assemble or export with the two MOSTA paper notebooks`

These panel-building files are in the repository release and are not installed with the Python package.

## Saved files

- Aligned data: `tutorial_outputs/mosta/preprocess/mosta_aligned.h5ad`
- Training directory: `tutorial_outputs/mosta/training`
- Downstream directory: `tutorial_outputs/mosta/downstream`